<a href="https://colab.research.google.com/github/garthajon/QuantFinanceIntro/blob/main/Liquiditystockscreener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import yfinance as yf
import numpy as np
import pandas as pd
import requests
import warnings

warnings.filterwarnings("ignore")

# ==========================================================
# PARAMETERS (UNCHANGED MODEL)
# ==========================================================

delta_days = 60
spread_ma_days = 5
volume_ma_days = 5
liquidity_rolling_days = 5

w_price = 1.0
w_spread = 1.0
w_volume = 1.5

threshold = 0.25


# ==========================================================
# SCRAPE S&P500 TICKERS
# ==========================================================

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)

table = pd.read_html(response.text)[0]

tickers = table["Symbol"].tolist()

tickers = [t.replace(".", "-") for t in tickers]

print(f"Loaded {len(tickers)} tickers")


# ==========================================================
# DOWNLOAD ALL DATA AT ONCE
# ==========================================================

data = yf.download(

    tickers,
    period="1y",
    interval="1d",
    auto_adjust=True,
    group_by="ticker",
    threads=True

)

print("Download complete")


# ==========================================================
# STANDARDIZATION FUNCTION
# ==========================================================

def standardize_series(series):

    max_abs = series.abs().max()

    if max_abs == 0 or np.isnan(max_abs):

        return series

    return series / max_abs


# ==========================================================
# LIQUIDITY CALCULATION
# ==========================================================

def compute_liquidity_score(df):

    df = df.copy()

    df["log_price"] = np.log(df["Close"])

    dt = df.index.to_series().diff(delta_days).dt.days

    df["price_fd"] = df["log_price"].diff(delta_days) / dt

    df["spread"] = df["High"] - df["Low"]

    df["log_spread"] = np.log(df["spread"])

    df["spread_fd"] = df["log_spread"].diff(delta_days) / dt

    df["spread_fd"] = df["spread_fd"].rolling(spread_ma_days, min_periods=1).mean()

    df["volume_log"] = np.log(df["Volume"] + 1)

    df["volume_fd"] = df["volume_log"].diff(delta_days) / dt

    df["volume_fd"] = df["volume_fd"].rolling(volume_ma_days, min_periods=1).mean()

    df = df.dropna()

    df["price_scaled"] = standardize_series(df["price_fd"])
    df["spread_scaled"] = standardize_series(df["spread_fd"])
    df["volume_scaled"] = standardize_series(df["volume_fd"])

    df["liquidity"] = (

        w_price * df["price_scaled"]
        - w_spread * df["spread_scaled"]
        + w_volume * df["volume_scaled"]

    ) / (w_price + w_spread + w_volume)

    df["liquidity"] = df["liquidity"].rolling(

        liquidity_rolling_days,
        min_periods=1

    ).mean()

    return df["liquidity"].iloc[-1]


# ==========================================================
# RUN SCREEN
# ==========================================================

results = []

for ticker in tickers:

    try:

        df = data[ticker].dropna()

        if len(df) < 100:
            continue

        score = compute_liquidity_score(df)

        results.append((ticker, score))

    except:

        continue


results_df = pd.DataFrame(results, columns=["Ticker", "LiquidityScore"])


# ==========================================================
# SCORE DISTRIBUTION
# ==========================================================

print("\nLiquidity Score Distribution\n")

print(results_df["LiquidityScore"].describe())


# ==========================================================
# THRESHOLD FILTER
# ==========================================================

bullish = results_df[results_df["LiquidityScore"] > threshold]\
    .sort_values("LiquidityScore", ascending=False)

bearish = results_df[results_df["LiquidityScore"] < -threshold]\
    .sort_values("LiquidityScore")


print("\n==============================")
print("BULLISH LIQUIDITY (>0.25)")
print("==============================\n")

print(bullish.to_string(index=False))


print("\n==============================")
print("BEARISH LIQUIDITY (<-0.25)")
print("==============================\n")

print(bearish.to_string(index=False))


# ==========================================================
# TOP SIGNALS
# ==========================================================

print("\n==============================")
print("TOP 15 BULLISH LIQUIDITY")
print("==============================\n")

print(results_df.sort_values("LiquidityScore", ascending=False).head(15))


print("\n==============================")
print("TOP 15 BEARISH LIQUIDITY")
print("==============================\n")

print(results_df.sort_values("LiquidityScore").head(15))

[                       0%                       ]

Loaded 503 tickers


[*********************100%***********************]  503 of 503 completed


Download complete

Liquidity Score Distribution

count    502.000000
mean      -0.008797
std        0.144047
min       -0.375449
25%       -0.101277
50%       -0.021773
75%        0.076565
max        0.512067
Name: LiquidityScore, dtype: float64

BULLISH LIQUIDITY (>0.25)

Ticker  LiquidityScore
  FANG        0.512067
   AES        0.492265
   CMS        0.440534
   OXY        0.389715
   KIM        0.377792
   COP        0.368168
   DVN        0.365323
   EOG        0.352803
   BKR        0.342857
   DUK        0.338708
   LYB        0.335394
    CF        0.331398
   AEP        0.328892
    BG        0.324789
   NRG        0.302422
   EXC        0.300087
   DOW        0.299228
  CTRA        0.282978
   CME        0.279273
   EIX        0.269263
   LNT        0.267675
    MU        0.267635
   APA        0.265867
    WM        0.264236
    NI        0.259770
   KMI        0.257438

BEARISH LIQUIDITY (<-0.25)

Ticker  LiquidityScore
   EFX       -0.375449
   CRH       -0.348159
   STZ 